# Monte Carlo Sampling for River-Aquifer Exchange

This notebook implements Monte Carlo sampling to explore the parameter space
of river-aquifer exchange dynamics using dimensionless analysis.

## Objectives
1. Generate parameter combinations for different drainage areas
2. Calculate dimensionless Pi numbers for each combination
3. Sample representative points using density-based clustering
4. Export samples for MODFLOW simulations

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.stats import hmean, gaussian_kde, mode
from sklearn.mixture import GaussianMixture
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
import os
import plotly.graph_objects as go

# Import project modules
import sys
sys.path.append('..')
from src import geomorphic_scaling as gs
from src import dimensionless_numbers as dn

## General Configuration

### Monte Carlo Simulation Parameters

In [ ]:
# GENERAL CONFIGURATION
# MONTE CARLO SIMULATION
# ------------------------

# Drainage areas to analyze (km²)
da_km2_values = [1000, 10000, 100000]

# FUNCTION FOR SCALED EVENT DURATION (t_d)
# ----------------------------------------
def design_td_seconds(da_km2, S=None, method="NRCS"):
    """
    Calculate event durations (t_d) consistent with catchment size.
    
    Parameters
    ----------
    da_km2 : float
        Drainage area (km²)
    S : float, optional
        Mean channel slope (m/m). If None, estimated from drainage area.
    method : str
        Method for duration calculation. Options: "NRCS"
    
    Returns
    -------
    list
        List of event durations [short, base, long] in seconds
    """
    # 1) Hydraulic length (Hack's Law)
    L_m = 1.4 * (da_km2 ** 0.6) * 1_000
    
    # 2) Channel slope
    if S is None:
        S = 0.001 * da_km2**-0.35
    
    # 3) Time of concentration using Kirpich formula (min → s)
    Tc_sec = 0.0195 * (L_m ** 0.77) * (S ** -0.385) * 60
    
    # 4) Base duration
    td_base = 0.6 * Tc_sec if method.upper() == "NRCS" else Tc_sec
    
    # 5) Duration range (±30%)
    td_values = [0.7 * td_base, td_base, 1.3 * td_base]
    
    return td_values

# Number of samples per catchment
n_samples = 2000

# Discrete values for Monte Carlo (empirical distributions)
dr_m_values = np.linspace(0.1, 0.2, 5)         # Streambed thickness ratio dr/d_bkf
kr_ka_values = [0.001, 0.005, 0.01, 0.05, 0.1] # Conductivity ratio Kr/Ka

## Event Duration Visualization

Control plot showing event duration (t_d) vs drainage area

In [ ]:
# --- EVENT DURATION (t_d) VS DRAINAGE AREA PLOT --------------------------
td_dict = {da: np.array(design_td_seconds(da)) / 86400
           for da in da_km2_values}  # Convert seconds to days

fig, ax = plt.subplots(figsize=(8, 6))
for i, da in enumerate(da_km2_values):
    ax.scatter(np.full_like(td_dict[da], i), td_dict[da], label=f"{da} km²")
ax.set_xticks(range(len(da_km2_values)))
ax.set_xticklabels([str(da) for da in da_km2_values])
ax.set_xlabel("Drainage Area (km²)")
ax.set_ylabel("Event Duration t_d (days)")
ax.set_title("Event Duration vs Catchment Size")
ax.legend(title="Catchment")
ax.grid(True)
plt.tight_layout()
plt.show()

## Monte Carlo Sampling

Generate parameter combinations and calculate dimensionless numbers

In [ ]:
all_results = []

for da_km2 in tqdm(da_km2_values, desc="Processing catchment sizes"):
    # Get channel geometry from geomorphic scaling
    w_bkf = gs.get_w_bkf(da_km2)
    d_bkf = gs.get_d_bkf(da_km2)
    base_S = gs.get_S(da_km2)
    dr_m = gs.get_dr(d_bkf)
    L = gs.get_L(w_bkf)
    H_0_array = 0.3 * d_bkf  # Base stage as fraction of bankfull depth
    t_d_values = design_td_seconds(da_km2, S=base_S, method="NRCS")
    
    for t_d in t_d_values:
        # Generate random parameter arrays
        kr_values = np.logspace(np.log10(1e-8), np.log10(1e-3), num=50000*4)
        C_values = np.logspace(np.log10(1e-4), np.log10(2e-1), num=50000*4)

        np.random.shuffle(kr_values)
        np.random.shuffle(C_values)

        # Calculate grid spacing based on conductance constraint
        dx = (C_values * dr_m) / (kr_values * w_bkf)
        
        # Filter for valid grid resolution (100 < nrow < 200)
        idx = ((L/dx) > 100) & ((L/dx) < 200)

        kr_clean = kr_values[idx]
        C_clean = C_values[idx]
        dx_clean = dx[idx]

        # Random permutation for sampling
        idx_random = np.random.permutation(len(kr_clean))

        kr = kr_clean[idx_random][:n_samples]
        C_raw = C_clean[idx_random][:n_samples]
        dx_raw = dx_clean[idx_random][:n_samples]

        # Recalculate for integer grid rows
        nrow = np.int32(L/dx_raw)
        dx = L / nrow
        C = kr * w_bkf * dx / dr_m
        
        # Conductivity ratio (Kr/Ka)
        kr_ka = np.logspace(1, 3, n_samples)
        np.random.shuffle(kr_ka)

        # Random sampling for other parameters
        Sy = np.random.uniform(0.10, 0.30, size=n_samples)  # Specific yield
        n = np.random.uniform(0.25, 0.45, size=n_samples)   # Manning coefficient
        B = np.random.uniform(10, 200, size=n_samples) * w_bkf   # Aquifer width (scaled to river width)
        D = np.random.uniform(1.0, 20.0, size=n_samples) * d_bkf # Aquifer depth (scaled to bankfull depth)

        for i in range(n_samples):
            if i % 100 == 0:
                print(f"  Simulation {i}/{n_samples} for catchment {da_km2}, t_d = {t_d / (24*3600):.1f} days")
            
            H_0 = H_0_array
            eta_0 = (d_bkf - H_0) / H_0
            H_p = eta_0 * H_0
            Kr = kr[i]
            Ka = Kr * kr_ka[i]
            Dr = dr_m

            # Calculate dimensionless numbers
            tc = dn.get_tc(Sy[i], Ka, B[i], d_bkf)
            gamma_d = dn.get_gamma_d(tc, t_d)
            pi1 = dn.get_lambda_v(tc, Kr, Dr)
            pi2 = dn.get_theta_r(Sy[i], w_bkf, B[i])
            pi3 = dn.get_chi(base_S, d_bkf, n[i], B[i], tc)
            rho_0 = dn.get_rho_0(H_p, H_0)
            zeta = dn.get_zeta(base_S, L, H_0)

            all_results.append({
                'da_km2': da_km2,
                'Sy': Sy[i],
                'Ka': Ka,
                'L': L,
                'B': B[i],
                'D': D[i],
                'dr': Dr,
                'Kr': Kr,
                'S': base_S,
                'n': n[i],
                't_d_days': t_d / (24 * 3600),
                'H_0': H_0,
                'eta_0': eta_0,
                'H_p': H_p,
                'w_bkf': w_bkf,
                'd_bkf': d_bkf,
                'tc': tc,
                'gamma_d': gamma_d,
                'pi1': pi1,
                'pi2': pi2,
                'pi3': pi3,
                'rho_0': rho_0,
                'zeta': zeta,
                'C': C[i],
                'dx': dx[i],
                'nrow': L/dx[i]
            })

results_df = pd.DataFrame(all_results)
print(f"\nTotal samples generated: {len(results_df)}")

## Cluster Analysis and Density-Based Sampling

Select representative samples based on density in dimensionless parameter space

In [ ]:
def sample_points_by_density_3d(points, n_samples):
    """
    Sample 3D points proportionally to their density using KDE.
    
    Parameters
    ----------
    points : ndarray
        Array of shape (n_points, 3) with coordinates [x, y, z]
    n_samples : int
        Number of points to select
    
    Returns
    -------
    tuple
        (sampled_points, sampled_indices)
    """
    kde = gaussian_kde(points.T)
    densities = kde(points.T)
    probabilities = densities / densities.sum()
    sampled_indices = np.random.choice(
        len(points),
        size=n_samples,
        p=probabilities,
        replace=True
    )
    return points[sampled_indices], sampled_indices


# Number of samples to select per cluster
N_SAMPLES = 50

all_samples = []  # List to accumulate partial DataFrames

for dakm in results_df['da_km2'].unique():
    data_tmp = results_df[results_df['da_km2'] == dakm]
    coords = data_tmp[['pi1', 'pi2', 'pi3']].values

    # Adjust if fewer points than N_SAMPLES
    n_req = min(N_SAMPLES, len(coords))
    sampled_pts, sampled_idx = sample_points_by_density_3d(coords, n_req)

    # Map sampled indices to original DataFrame positions
    original_indices = data_tmp.index.values[sampled_idx]

    # Extract selected rows
    df_part = results_df.loc[original_indices, :].reset_index(drop=True)

    all_samples.append(df_part)

# Concatenate all samples and export
df_samples = pd.concat(all_samples, ignore_index=True)
df_samples.to_csv("../data/output/sample_points.csv", index=False)

print(f"Selected {len(df_samples)} samples for simulation")
print(f"Samples saved to: ../data/output/sample_points.csv")

## Visualization of Sampled Points

In [ ]:
# 3D scatter plot of sampled points in dimensionless space
fig = go.Figure()

for dakm in df_samples['da_km2'].unique():
    subset = df_samples[df_samples['da_km2'] == dakm]
    fig.add_trace(go.Scatter3d(
        x=np.log10(subset['pi1']),
        y=np.log10(subset['pi2']),
        z=np.log10(subset['pi3']),
        mode='markers',
        name=f'{dakm} km²',
        marker=dict(size=5)
    ))

fig.update_layout(
    title='Sampled Points in Dimensionless Parameter Space',
    scene=dict(
        xaxis_title='log₁₀(π₁) - Lambda_v',
        yaxis_title='log₁₀(π₂) - Theta_r',
        zaxis_title='log₁₀(π₃) - Chi'
    ),
    width=900,
    height=700
)

fig.show()

## Summary Statistics

In [ ]:
# Display summary statistics for sampled parameters
print("Summary Statistics for Sampled Points")
print("=" * 50)

for col in ['Ka', 'Kr', 'Sy', 'B', 'D', 'gamma_d', 'pi1', 'pi2', 'pi3']:
    print(f"\n{col}:")
    print(f"  Min:    {df_samples[col].min():.4e}")
    print(f"  Max:    {df_samples[col].max():.4e}")
    print(f"  Mean:   {df_samples[col].mean():.4e}")
    print(f"  Median: {df_samples[col].median():.4e}")